# Python Descriptors and __slots__
Covers: descriptor protocol (__get__/__set__/__delete__/__set_name__), data vs non-data descriptors, __slots__ memory optimization

## 1. Descriptor Protocol Basics

In [ ]:
class LoggedDescriptor:
    """Descriptor that logs all attribute access."""
    
    def __set_name__(self, owner, name):
        self.name = name
        self.private_name = f'_{name}'
        print(f'[__set_name__] {owner.__name__}.{name}')
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            print(f'[__get__] Accessed on class {objtype.__name__}')
            return self
        value = getattr(obj, self.private_name, None)
        print(f'[__get__] {obj.__class__.__name__}.{self.name} = {value!r}')
        return value
    
    def __set__(self, obj, value):
        print(f'[__set__] {obj.__class__.__name__}.{self.name} = {value!r}')
        setattr(obj, self.private_name, value)
    
    def __delete__(self, obj):
        print(f'[__delete__] {obj.__class__.__name__}.{self.name}')
        if hasattr(obj, self.private_name):
            delattr(obj, self.private_name)

class MyClass:
    x = LoggedDescriptor()  # __set_name__ called here
    y = LoggedDescriptor()

print('\n--- Instance operations ---')
obj = MyClass()
obj.x = 10      # __set__
obj.y = 20      # __set__
val = obj.x     # __get__
del obj.x       # __delete__

print('\n--- Class access ---')
desc = MyClass.x  # __get__ with obj=None

## 2. Data vs Non-Data Descriptors

In [ ]:
class DataDescriptor:
    """Data descriptor — has __set__, takes priority over instance __dict__."""
    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return obj.__dict__.get('_data_val', 'data_default')
    
    def __set__(self, obj, value):
        obj.__dict__['_data_val'] = f'data:{value}'

class NonDataDescriptor:
    """Non-data descriptor — only __get__, instance __dict__ wins."""
    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return 'from_non_data_descriptor'

class Demo:
    data = DataDescriptor()
    non_data = NonDataDescriptor()

obj = Demo()

# Data descriptor: __set__ is called
obj.data = 42
print('data (via descriptor):', obj.data)  # 'data:42'
print('instance __dict__:', obj.__dict__)

# Non-data descriptor: instance __dict__ takes priority!
print('\nnon_data before override:', obj.non_data)  # from descriptor
obj.__dict__['non_data'] = 'from_instance'  # bypass descriptor
print('non_data after override:', obj.non_data)  # from instance!

print('\nLookup order:')
print('1. Data descriptors (class)')
print('2. Instance __dict__')
print('3. Non-data descriptors (class)')

## 3. Practical Descriptor — Type Validation

In [ ]:
class Validator:
    """Base descriptor for validated attributes."""
    
    def __set_name__(self, owner, name):
        self.name = name
        self.private_name = f'_{name}'
    
    def __get__(self, obj, objtype=None):
        if obj is None: return self
        return getattr(obj, self.private_name, None)
    
    def __set__(self, obj, value):
        value = self.validate(value)
        setattr(obj, self.private_name, value)
    
    def validate(self, value):
        return value

class PositiveNumber(Validator):
    def validate(self, value):
        if not isinstance(value, (int, float)):
            raise TypeError(f"'{self.name}' must be a number, got {type(value).__name__}")
        if value <= 0:
            raise ValueError(f"'{self.name}' must be positive, got {value}")
        return float(value)

class NonEmptyString(Validator):
    def validate(self, value):
        if not isinstance(value, str):
            raise TypeError(f"'{self.name}' must be a string")
        stripped = value.strip()
        if not stripped:
            raise ValueError(f"'{self.name}' cannot be empty")
        return stripped

class RangedInt(Validator):
    def __init__(self, min_val, max_val):
        self.min_val = min_val
        self.max_val = max_val
    
    def validate(self, value):
        if not isinstance(value, int):
            raise TypeError(f"'{self.name}' must be an int")
        if not (self.min_val <= value <= self.max_val):
            raise ValueError(f"'{self.name}' must be in [{self.min_val}, {self.max_val}]")
        return value

class Product:
    name = NonEmptyString()
    price = PositiveNumber()
    quantity = RangedInt(0, 10000)
    
    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity
    
    @property
    def total_value(self):
        return self.price * self.quantity
    
    def __repr__(self):
        return f'Product({self.name!r}, ${self.price:.2f}, qty={self.quantity})'

p = Product('Widget', 9.99, 100)
print('Product:', p)
print('Total value: $', p.total_value)

# Test validation
test_cases = [
    ('price', -5, ValueError),
    ('name', '', ValueError),
    ('quantity', 99999, ValueError),
    ('price', 'free', TypeError),
]

for attr, value, expected_exc in test_cases:
    try:
        setattr(p, attr, value)
        print(f'ERROR: Should have raised {expected_exc.__name__}')
    except expected_exc as e:
        print(f'{expected_exc.__name__} for {attr}={value!r}: {e}')

## 4. __slots__ — Memory Optimization

In [ ]:
import sys
import tracemalloc

class RegularPoint:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

class SlottedPoint:
    __slots__ = ('x', 'y', 'z')
    
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

# Single instance size
r = RegularPoint(1.0, 2.0, 3.0)
s = SlottedPoint(1.0, 2.0, 3.0)

print('Single instance sizes:')
print(f'  Regular: {sys.getsizeof(r)} bytes (object) + {sys.getsizeof(r.__dict__)} bytes (__dict__)')
print(f'  Slotted: {sys.getsizeof(s)} bytes (no __dict__)')
print(f'  Regular has __dict__: {hasattr(r, "__dict__")}')
print(f'  Slotted has __dict__: {hasattr(s, "__dict__")}')

# Memory with many instances
N = 100_000

tracemalloc.start()
regular_list = [RegularPoint(float(i), float(i), float(i)) for i in range(N)]
_, regular_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
slotted_list = [SlottedPoint(float(i), float(i), float(i)) for i in range(N)]
_, slotted_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'\n{N:,} instances:')
print(f'  Regular: {regular_peak / 1024 / 1024:.1f} MB')
print(f'  Slotted: {slotted_peak / 1024 / 1024:.1f} MB')
print(f'  Memory savings: {(1 - slotted_peak/regular_peak)*100:.0f}%')

# Slotted class restrictions
try:
    s.w = 4.0  # AttributeError — not in __slots__
except AttributeError as e:
    print(f'\nAttributeError: {e}')

## 5. Lazy Property Descriptor

In [ ]:
import time

class lazy_property:
    """Non-data descriptor that computes value once and caches it."""
    
    def __init__(self, func):
        self.func = func
        self.__doc__ = func.__doc__
    
    def __set_name__(self, owner, name):
        self.name = name
    
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        # Compute and cache in instance __dict__
        # (non-data descriptor — instance __dict__ wins on next access)
        value = self.func(obj)
        obj.__dict__[self.name] = value  # cache
        return value

class DataProcessor:
    def __init__(self, data):
        self.data = data
    
    @lazy_property
    def sorted_data(self):
        """Expensive sort — computed once."""
        print('  Computing sorted_data...')
        time.sleep(0.1)  # simulate expensive computation
        return sorted(self.data)
    
    @lazy_property
    def statistics(self):
        """Expensive statistics — computed once."""
        print('  Computing statistics...')
        data = self.sorted_data  # uses cached value
        n = len(data)
        return {
            'min': data[0],
            'max': data[-1],
            'mean': sum(data) / n,
            'median': data[n // 2],
        }

import random
data = [random.randint(1, 100) for _ in range(1000)]
processor = DataProcessor(data)

print('First access (computes):')
s1 = processor.sorted_data

print('Second access (cached):')
s2 = processor.sorted_data  # no recomputation!

print('Statistics (uses cached sorted_data):')
stats = processor.statistics
print(f'  {stats}')

print('\nCached in __dict__:', 'sorted_data' in processor.__dict__)